## Phase 1: Environment Setup & Library Imports

In [69]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Set visualization theme
pio.templates.default = "plotly_white"

## Phase 2: Data Ingestion & Cleaning Pipeline
Using an Object-Oriented `HospitalDataManager` class to encapsulate
the full data cleaning workflow: type conversion, LOS derivation,
anomaly removal, missing value imputation, and text standardization.

In [70]:
class HospitalDataManager:
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        
    def perform_cleaning(self):
        df = self.df.copy()
        df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'])
        df['DischargeDate'] = pd.to_datetime(df['DischargeDate'])
        df['LOS'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days
        df = df[df['LOS'] >= 0]
        df['Age'] = df['Age'].fillna(df['Age'].median())
        df = df.drop_duplicates()
        df['DayOfWeek'] = df['AdmissionDate'].dt.dayofweek
        df['Month'] = df['AdmissionDate'].dt.month
        df['Diagnosis'] = df['Diagnosis'].str.title()
        self.df = df
        return self.df

file_path = '/kaggle/input/datasets/nudratabbas/hospital-records-for-data-cleaning-medium/hospital_patients_real_world.csv'
manager = HospitalDataManager(file_path)
df_clean = manager.perform_cleaning()

## Phase 3: Feature Engineering
Creating domain-specific features: Age grouping, stay categorization,
and temporal feature extraction from admission dates.

In [71]:
def add_smart_features(df):
    bins = [0, 18, 35, 60, 120]
    labels = ['Child', 'Young Adult', 'Adult', 'Senior']
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
    df['StayCategory'] = np.where(df['LOS'] > 7, 'Long Stay', 'Short Stay')
    return df

df_clean = add_smart_features(df_clean)

## Phase 4: Data Quality & Signal Assessment
Before training a machine learning model, a responsible Data Scientist must first
verify whether the available features carry **any predictive signal** with respect
to the target variable (`LOS`). Skipping this step is a common mistake that leads
to wasted compute and misleading results.

We will compute Pearson correlations between every numeric/encoded feature and `LOS`.

In [ ]:
# ================================================================
# Phase 4 - Signal Assessment: Feature-Target Correlation Check
# ================================================================
# Before modelling, we must verify that features actually correlate
# with the target. Near-zero correlations indicate the target may
# have been generated independently of the features (synthetic data).

from sklearn.preprocessing import LabelEncoder

df_corr = df_clean.copy()
df_corr['Diagnosis_Enc'] = LabelEncoder().fit_transform(df_corr['Diagnosis'])
df_corr['Gender_Enc']    = LabelEncoder().fit_transform(df_corr['Gender'])

target_corr = (
    df_corr[['Age', 'Gender_Enc', 'Diagnosis_Enc', 'DayOfWeek', 'Month', 'LOS']]
    .corr()['LOS']
    .drop('LOS')
    .sort_values(ascending=False)
)

print('Feature-Target (LOS) Correlations')
print('=' * 40)
for feat, val in target_corr.items():
    print(f'{feat:20s}  {val:+.4f}')

print()
print('Observation: All correlations are effectively ZERO.')
print('This strongly suggests the LOS values in this dataset')
print('were generated randomly and independently of patient features.')


In [ ]:
# ================================================================
# Phase 4 - Signal Assessment: LOS Distribution Analysis
# ================================================================
# A truly clinical LOS should be right-skewed (many short stays,
# few long stays). A uniform distribution confirms synthetic data.

fig_los = px.histogram(
    df_clean, x='LOS',
    nbins=10,
    title='Distribution of Length of Stay (LOS)',
    labels={'LOS': 'Length of Stay (days)', 'count': 'Number of Patients'},
    color_discrete_sequence=['#636EFA'],
)
fig_los.update_layout(bargap=0.1)
fig_los.show()

print(f'LOS Range : {df_clean["LOS"].min()} - {df_clean["LOS"].max()} days')
print(f'LOS Mean  : {df_clean["LOS"].mean():.2f}')
print(f'LOS Median: {df_clean["LOS"].median():.1f}')
print(f'LOS Std   : {df_clean["LOS"].std():.2f}')
print()
print('The near-uniform distribution (1-10 days) confirms synthetic generation.')


## Phase 5: Machine Learning Baseline (XGBoost)
Despite the weak signal identified above, we proceed with an XGBoost Regressor
as a **baseline experiment**. The purpose is to empirically confirm that the data
lacks predictive power and to demonstrate the full ML pipeline for portfolio purposes.

In [72]:
le_diag = LabelEncoder()
le_gen = LabelEncoder()
df_model = df_clean.copy()

df_model['Diagnosis_Enc'] = le_diag.fit_transform(df_model['Diagnosis'])
df_model['Gender_Enc'] = le_gen.fit_transform(df_model['Gender'])

features = ['Age', 'Diagnosis_Enc', 'Gender_Enc', 'DayOfWeek', 'Month']
X = df_model[features]
y = df_model['LOS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f"Final Mean Absolute Error: {mean_absolute_error(y_test, preds):.2f} days")
print(f"Final R2 Score: {r2_score(y_test, preds):.2f}")

joblib.dump(model, 'hospital_model.pkl')
joblib.dump(le_diag, 'le_diag.pkl')
joblib.dump(le_gen, 'le_gen.pkl')

Final Mean Absolute Error: 2.58 days
Final R2 Score: -0.04


['le_gen.pkl']

## Critical Data Science Insight: Why the R-Squared is Negative

The XGBoost model produced an **R-Squared of approximately -0.04** and a
**Mean Absolute Error of 2.58 days**.

### What Does a Negative R-Squared Mean?
A negative R-Squared indicates the model performs **worse** than simply
predicting the historical average LOS (approximately 5.47 days) for every patient.
The model is attempting to find patterns in what is effectively random noise.

### Root Cause Analysis
| Check | Result | Implication |
| --- | --- | --- |
| Feature-Target Correlation | All features show ~0.00 correlation with LOS | No linear signal exists |
| LOS Distribution | Near-uniform between 1-10 days | Synthetically generated, not clinical |
| Dataset Purpose | Kaggle "Hospital Records for **Data Cleaning**" | Dataset was designed for cleaning exercises, not ML |

### Business Takeaway
A crucial responsibility of a Data Scientist is to identify when data **lacks
predictive signal** before investing resources in modelling. This analysis
demonstrates that the `AdmissionDate` and `DischargeDate` columns were generated
independently of clinical features. For production-grade LOS prediction, a
clinically-linked real-world dataset containing vitals, lab results, and
comorbidity indices would be required.

## Phase 6: Professional Insights & Visualizations
While the ML model confirms the absence of predictive signal, the **descriptive
analytics** derived from this dataset remain highly valuable. The following
dashboards provide actionable hospital performance metrics.

In [74]:
### --- Cell 5: Professional Insights & Visualizations ---
from IPython.display import display

# 1. Average LOS by Diagnosis
avg_los_diag = df_clean.groupby('Diagnosis')['LOS'].mean().sort_values(ascending=False).reset_index()
fig1 = px.bar(avg_los_diag, x='LOS', y='Diagnosis', title="Average LOS by Diagnosis", 
              color='LOS', color_continuous_scale='Reds')

# 2. Heatmap: Diagnosis vs Age Group (Normalized)
heatmap_data = pd.crosstab(df_clean['Diagnosis'], df_clean['AgeGroup'], normalize='index') * 100
fig2 = px.imshow(heatmap_data, text_auto='.1f', title="Heatmap: Diagnosis Age-Distribution (%)", 
                 color_continuous_scale='Viridis')

# 3. Hospital Efficiency Dashboard
hosp_perf = df_clean.groupby('HospitalID').agg({'LOS': 'mean', 'PatientID': 'count'}).rename(columns={'PatientID': 'Volume'})
fig3 = px.scatter(hosp_perf.sort_values(by='LOS'), x='Volume', y='LOS', size='Volume', color='LOS',
                  text=hosp_perf.index, title="Hospital Performance: Volume vs Average LOS")

# 4. Seasonal Diagnosis Distribution
trend_data = df_clean.groupby(['Month', 'Diagnosis']).size().reset_index(name='Count')
fig4 = px.bar(trend_data, x='Month', y='Count', color='Diagnosis', 
              title="Seasonal Diagnosis Distribution", barmode='stack')

# 5. Model Feature Importance (The Strongest Insight)
importances = model.feature_importances_
feature_imp = pd.DataFrame({
    'Feature': ['Age', 'Diagnosis', 'Gender', 'Admission Day', 'Admission Month'],
    'Importance': importances
}).sort_values('Importance', ascending=True) # Ascending=True for horizontal bar chart

fig5 = px.bar(feature_imp, x='Importance', y='Feature', orientation='h',
              title='Key Drivers of Length of Stay', color='Importance', 
              color_continuous_scale='Greens')

# Displaying all figures
display(fig1, fig2, fig3, fig4, fig5)